# 03 Features: pool and label per cut-off

Drafted with an AI assistant, see docs/ai_log.md.

Inputs: `data/processed/buildings_clean.parquet`, `data/processed/pv_fr.parquet`.
Output: `data/processed/model_table.parquet` (one row per pool building and cut-off, with features, `y` and `y_2024`).

For each cut-off T in `helpers.CUTOFFS`: pool = buildings without PV by T, not new builds, with enough suitable roof. Label `y` = first PV in the two years after T. For the test split, `y_2024` uses 2024 only.

Neighbour features: for each cut-off, `has_pv` is the first PV date <= T over all population buildings; `nbr_n_R` is the number of other population buildings within R metres (R = 100, 250) and `nbr_pv_share_R` the share of them with `has_pv` (0 when none).


Tariff features: `tariff_total` is the H4 electricity price (Rp/kWh) of year T + 1 (2022 for train, 2024 for test), joined on `ggdenr` = `bfs_nr`. Municipalities without a tariff row for that year get the canton median of the year and `tariff_filled` = 1. `tariff_dev` = `tariff_total` minus the canton median of the year, the model feature (the price level shifts between cut-offs). The canton median is the median over the municipality rows of `tariffs_fr.parquet` for that year.

Id columns: `ggdenr` (municipality number, groups for cross-validation), `gkode`, `gkodn` (LV95 coordinates, for grid cells and maps). They are identifiers, not model features; `tariff_total` and `tariff_filled` are also not features.

The pool-filter table (`figures/table_pool_filters.tex`) follows the model table: how many at-risk buildings each filter removes and the adoption rate of each group.

In [ ]:
import helpers
import pandas as pd

In [ ]:
buildings = pd.read_parquet(helpers.PROCESSED / "buildings_clean.parquet")
pv = pd.read_parquet(helpers.PROCESSED / "pv_fr.parquet")
first_pv = helpers.first_pv_date(pv)
print(len(buildings), "buildings,", len(first_pv), "buildings with a dated PV plant")
tariffs = pd.read_parquet(helpers.PROCESSED / "tariffs_fr.parquet")
# Canton median of the year: median over the municipality rows of tariffs_fr (one row per
# municipality and year). Used both to fill missing municipalities and as the reference level.
canton_median = tariffs.groupby("year")["total"].median()
print(canton_median.loc[[2022, 2024]].to_dict())

Build the pool and the label for each split.

In [ ]:
features = [
    "yield_kwh",
    "suitable_area_m2",
    "roof_area_m2",
    "best_class",
    "n_surfaces",
    "gkat",
    "gklas",
    "gbaup",
    "garea",
    "gastw",
    "ganzwhg",
]
parts = []
for split, cutoff in helpers.CUTOFFS.items():
    start = pd.Timestamp(cutoff)
    end = start + pd.DateOffset(years=helpers.HORIZON_YEARS)
    pool = helpers.build_pool(buildings, first_pv, cutoff)
    part = pool[["egid", "ggdenr", "gkode", "gkodn"] + features].copy()
    part.insert(1, "cutoff", cutoff)
    part.insert(2, "split", split)
    part.insert(3, "y", helpers.make_label(pool["egid"], first_pv, start, end))
    # Robustness label: first PV in the first year only (test split only)
    part["y_2024"] = pd.array([pd.NA] * len(part), dtype="Int64")
    if split == "test":
        end_one_year = start + pd.DateOffset(years=1)
        part["y_2024"] = helpers.make_label(pool["egid"], first_pv, start, end_one_year).astype(
            "Int64"
        )
    # Neighbour PV share: PV dated <= T, measured on all population buildings (not only the pool)
    has_pv = helpers.has_pv_by(buildings["egid"], first_pv, cutoff)
    xy = buildings[["gkode", "gkodn"]].to_numpy()
    nbr = pd.DataFrame({"egid": buildings["egid"]})
    for radius in [100, 250]:
        nbr[f"nbr_n_{radius}"], nbr[f"nbr_pv_share_{radius}"] = helpers.neighbour_pv_share(
            xy, has_pv, radius
        )
    part = part.merge(nbr, on="egid", how="left", validate="one_to_one")
    # Tariff known at T: the price of year T + 1; missing municipalities get the canton median
    tariff_year = start.year + 1
    price = tariffs[tariffs["year"] == tariff_year].set_index("bfs_nr")["total"]
    part["tariff_total"] = part["ggdenr"].map(price)
    part["tariff_filled"] = part["tariff_total"].isna().astype(int)
    part["tariff_total"] = part["tariff_total"].fillna(canton_median[tariff_year])
    part["tariff_dev"] = (part["tariff_total"] - canton_median[tariff_year]).round(3)
    parts.append(part)
    print(split, "label window", start.date() + pd.Timedelta(days=1), "to", end.date())

neighbour_columns = ["nbr_n_100", "nbr_pv_share_100", "nbr_n_250", "nbr_pv_share_250"]
id_columns = ["ggdenr", "gkode", "gkodn"]
tariff_columns = ["tariff_total", "tariff_filled", "tariff_dev"]
order = (
    ["egid", "cutoff", "split", "y", "y_2024"]
    + features
    + neighbour_columns
    + id_columns
    + tariff_columns
)
model_table = pd.concat(parts, ignore_index=True)[order]
model_table = model_table.sort_values(["cutoff", "egid"]).reset_index(drop=True)

In [ ]:
assert not model_table.duplicated(["egid", "cutoff"]).any()
for split, part in model_table.groupby("split"):
    print(
        split,
        "pool:",
        len(part),
        "positives:",
        int(part["y"].sum()),
        "base rate:",
        round(part["y"].mean(), 4),
    )
test = model_table[model_table["split"] == "test"]
print(
    "test y_2024 positives:",
    int(test["y_2024"].sum()),
    "base rate:",
    round(test["y_2024"].mean(), 4),
)
model_table.to_parquet(helpers.PROCESSED / "model_table.parquet", index=False)

## Pool filters

For each cut-off, the at-risk buildings (no PV at the cut-off) are split by the first filter that removes them, in the order new build, no roof record, suitable roof under 20 m2 (`helpers.pool_status`, the same rules as `helpers.build_pool`). The rows add up to the at-risk row. Output: `figures/table_pool_filters.tex`.

In [ ]:
# Count buildings and adopters in each group, per cut-off
labels = {
    "at_risk": "At risk (no PV at the cut-off)",
    "new_build": "New build",
    "no_roof": "No roof record",
    "small_roof": "Suitable roof under 20 m$^2$",
    "pool": "Pool",
}
stats = {}
for split, cutoff in helpers.CUTOFFS.items():
    start = pd.Timestamp(cutoff)
    end = start + pd.DateOffset(years=helpers.HORIZON_YEARS)
    status = helpers.pool_status(buildings, first_pv, cutoff)
    adopted = helpers.make_label(buildings["egid"], first_pv, start, end)
    at_risk = status != "has_pv"
    rows = {"at_risk": at_risk}
    rows.update({k: status == k for k in ["new_build", "no_roof", "small_roof", "pool"]})
    stats[split] = {k: (int(m.sum()), int(adopted[m].sum())) for k, m in rows.items()}
    # The pool row must equal the model table, and the filters must add up to the at-risk row
    in_table = model_table[model_table["split"] == split]
    assert stats[split]["pool"] == (len(in_table), int(in_table["y"].sum()))
    parts_n = sum(stats[split][k][0] for k in ["new_build", "no_roof", "small_roof", "pool"])
    assert parts_n == stats[split]["at_risk"][0]
for split in stats:
    for k, (n, a) in stats[split].items():
        print(split, k, n, a, round(100 * a / n, 1))

In [ ]:
# Plain LaTeX tabular with fixed formatting, to be \\input in the paper
lines = [
    "\\begin{tabular}{lrrrrrr}",
    "\\toprule",
    " & \\multicolumn{3}{c}{Train (cut-off 2021)} & \\multicolumn{3}{c}{Test (cut-off 2023)} \\\\",
    "\\cmidrule(lr){2-4} \\cmidrule(lr){5-7}",
    " & Buildings & Adopters & Rate (\\%) & Buildings & Adopters & Rate (\\%) \\\\",
    "\\midrule",
]
for key, label in labels.items():
    cells = []
    for split in ["train", "test"]:
        n, a = stats[split][key]
        cells += [f"{n:,}", f"{a:,}", f"{100 * a / n:.1f}"]
    row = f"{label} & " + " & ".join(cells) + " \\\\"
    lines.append(row.replace(",", "{,}"))
    if key == "at_risk":
        lines.append("\\midrule")
lines += ["\\bottomrule", "\\end{tabular}"]
(helpers.FIGURES / "table_pool_filters.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))

## Register lag

Separate from the model table. Compares the plant register of 2026-09-14 (published 2026-09-16) with the current one: how many first PV installations per start year are counted in each, on the population buildings. A first installation is the earliest start date of a building (`helpers.first_pv_date`). Output: `figures/table_register_lag.tex`.

In [ ]:
# Older snapshot, same columns as pv_fr.parquet
pv_sep = pd.read_parquet(helpers.PROCESSED / "pv_fr_2026_09.parquet")
first_sep = helpers.first_pv_date(pv_sep)
print("latest start date, older snapshot:", pv_sep["start_date"].max().date())
print("latest start date, current file:", pv["start_date"].max().date())

In [ ]:
# First installations per start year, population buildings only
population = set(buildings["egid"])
rows = []
for year in [2022, 2023, 2024, 2025]:
    counts = []
    for first in [first_sep, first_pv]:
        in_population = first[first.index.isin(population)]
        counts.append(int((in_population.dt.year == year).sum()))
    old, new = counts
    rows.append(
        {"year": year, "old": old, "new": new, "diff": new - old, "pct": 100 * (new - old) / new}
    )
lag = pd.DataFrame(rows)
print(lag.to_string(index=False))

In [ ]:
# Plain LaTeX tabular with fixed formatting, to be \\input in the paper
lines = [
    "\\begin{tabular}{rrrrr}",
    "\\toprule",
    "Start year & September file & November file & Difference & Difference (\\%) \\\\",
    "\\midrule",
]
for r in lag.itertuples():
    lines.append(
        f"{r.year} & {r.old:,} & {r.new:,} & {r.diff:,} & {r.pct:.1f} \\\\".replace(",", "{,}")
    )
lines += ["\\bottomrule", "\\end{tabular}"]
(helpers.FIGURES / "table_register_lag.tex").write_text("\n".join(lines) + "\n")
print("\n".join(lines))